# Preliminary Phase 1 on Kaggle: trajectory balance (GFlowNet) against GRPO

This runs the Phase 1 comparison at a smaller scale, inside one free Kaggle session. Qwen3-1.7B learns to fix a wrong step 2 in a GSM8K answer, and half of each group of answers continues after " Wait".

- `PAIR = "main"` runs T5 (trajectory balance) against T4 (GRPO), both learning from the nudged answers. Run this first.
- `PAIR = "own"` runs T3 against T2, on the model's own answers only. Run it in a second session.
- `PAIR = "subtb"` runs sub-trajectory balance with the log reward of Hu et al. (ICLR 2024), with and without the "Wait" answers, for 80 steps. `docs/subtb-pilot.md` lists every setting.

The two runs share the session, one on each T4. If Kaggle's 12-hour limit would cut a run off, the run stops early, evaluates and saves. `docs/kaggle-prelim.md` lists what is smaller than in Phase 1.

**Settings:** Accelerator **GPU T4 x2**, Internet **on**.

**Run it with Save Version, then Save & Run All (Commit)**, so it keeps running after you close the browser. When it finishes, download `bundles/prelim-*.tar.gz` from the version's Output tab.

In [ ]:
PAIR = "main"   # "main": T5 against T4. "own": T3 against T2.
                # "subtb": sub-trajectory balance with and without "Wait" answers (docs/subtb-pilot.md).
REPO_URL = "https://github.com/amitkov-sudo/grpo-vs-gfn-self-correction.git"

!test -d repo || git clone -q $REPO_URL repo
%cd repo
!git pull -q && git log -1 --format="commit %h, %cd"

In [ ]:
# Installs the requirements, runs the offline checks, then trains both runs side by side (up to 11.5 hours).
!PAIR=$PAIR bash scripts/kaggle_prelim.sh

In [ ]:
import glob, json, os
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

display(Markdown(open("outputs/prelim-summary.md").read()))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for path in sorted(glob.glob("outputs/prelim-*/log.jsonl")):
    evals = [r["eval"] for r in map(json.loads, open(path)) if "eval" in r]
    run = os.path.basename(os.path.dirname(path))
    steps = [e["step"] for e in evals]
    ax1.plot(steps, [e["fix_rate"] for e in evals], marker="o", label=run)
    ax2.plot(steps, [e["false_alarms"] for e in evals], marker="o", label=run)
ax1.set(xlabel="training step", ylabel="fix rate on 60 wrong starts (greedy)")
ax2.set(xlabel="training step", ylabel="false alarms on 60 correct starts")
ax1.legend()
plt.tight_layout()
plt.show()